In [225]:
import pandas as pd 
import numpy as np 

## Import the data sets 


In [226]:
raw = pd.read_excel('../data/raw/HMIS Data by Province Tonga 28.09.26.xls', header = None)

In [227]:
# Locate the row whose first cell is "dataname" and use it as the header
hdr = raw.index[raw.iloc[:, 0].astype(str).str.strip().str.lower() == "dataname"][0]
df = raw.iloc[hdr + 1:].copy()
df.columns = raw.iloc[hdr].astype(str).str.strip()
df = df.rename(columns={df.columns[0]: "dataname"}).reset_index(drop=True)

# Cleaning 

#### filter out rows of choice

In [228]:
# Facility-reported case counts
case_rows = [
    "Cholera (HIA1)", "Cholera under 5 years - total", "Cholera 5 years and older - total",
    "Measles", "Measles under 5 years - total", "Measles 5 years and older - total",
    "Typhoid fever", "Typhoid fever under 5 years - total", "Typhoid fever 5 years and older - total",
]
facility = df[df["dataname"].isin(case_rows)]

# Community-reported suspected cases (regex avoids the stray "�" character in one label)
community = df[df["dataname"].str.contains(
    r"^Com_Number of.*suspected (?:Cholera|measles) cases", regex=True, na=False
)]

df_indicator = pd.concat([facility, community]).set_index("dataname")


In [229]:
df_indicator.head(1)

1,Zambia Ministry of Health January 2021,Zambia Ministry of Health February 2021,Zambia Ministry of Health March 2021,Zambia Ministry of Health April 2021,Zambia Ministry of Health May 2021,Zambia Ministry of Health June 2021,Zambia Ministry of Health July 2021,Zambia Ministry of Health August 2021,Zambia Ministry of Health September 2021,Zambia Ministry of Health October 2021,...,Lusaka June 2021,Lusaka July 2021,Lusaka August 2021,Lusaka September 2021,Lusaka October 2021,Lusaka November 2021,Lusaka December 2021,Lusaka January 2022,Lusaka February 2022,Lusaka March 2022
dataname,,,,,,,,,,,,,,,,,,,,,
Cholera 5 years and older - total,2,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### save df_indicator as it's own file

In [230]:
df_indicator.to_csv("../data/interim/indicator_data.csv", index=True)

##### change formt from wide to long

In [231]:
df_indicator = df.melt(
    id_vars="dataname",
    var_name="location_month",
    value_name="value"
)

##### separate location and month

In [232]:
df_indicator[["province", "month"]] = df_indicator["location_month"].str.extract(
    r"^(.*?)\s+([A-Za-z]+\s+\d{4})$"
)

In [233]:
df_indicator.head()

,dataname,location_month,value,province,month
0,Cholera 1-4 years CFR (%),Zambia Ministry of Health January 2021,NaN,Zambia Ministry of Health,January 2021
1,Cholera 1-4 years incidence (1K),Zambia Ministry of Health January 2021,NaN,Zambia Ministry of Health,January 2021
2,Cholera 5 years and older CFR (%),Zambia Ministry of Health January 2021,NaN,Zambia Ministry of Health,January 2021
3,Cholera 5 years and older incidence (1K),Zambia Ministry of Health January 2021,0,Zambia Ministry of Health,January 2021
4,Cholera 5 years and older - total,Zambia Ministry of Health January 2021,2,Zambia Ministry of Health,January 2021


In [234]:
# Convert month into a proper datetime
df_indicator["month"] = pd.to_datetime(
    df_indicator["month"],
    format="%B %Y",
    errors="coerce"
)

In [235]:
#add a disease type column 
def get_disease(indicator):

    indicator = indicator.lower()

    if "cholera" in indicator:
        return "cholera"   
    elif "measles" in indicator:
        return "measles"
    elif "typhoid" in indicator:
        return "typhoid"
    else: 
        return 'other'

df_indicator["disease"] = df_indicator['dataname'].apply(get_disease)

In [236]:
# add a case type column 
def get_case_type(indicator):

    indicator = indicator.lower()

    if "suspected" in indicator:
        return "suspected"
    else:
        return "reported"

df_indicator["case_type"] = df_indicator["dataname"].apply(get_case_type)

In [237]:
# make the value column in to numeric values
df_indicator["value"] = pd.to_numeric(
    df_indicator["value"],
    errors="coerce"
)

In [238]:
# add an age group column
def get_age_group(indicator):

    indicator = indicator.lower()

    if "under 5" in indicator:
        return "Under 5"

    elif "5 years and older" in indicator:
        return "5 years and older"

    else:
        return "All ages"


df_indicator["age_group"] = df_indicator["dataname"].apply(get_age_group)

In [239]:
# rename the data type column to indicator
df_indicator.rename(columns={"dataname": "indicator"}, inplace=True)

In [240]:
# drop the location month column 
df_indicator.drop(columns=['location_month'], inplace=True)

In [241]:
# rearrange the data
df_indicator = df_indicator[
    [
        "province",
        "month",
        "disease",
        "case_type",
        "age_group",
        "indicator",
        "value"
    ]
]

#### save processed data

In [247]:
df_indicator.to_csv('../data/processed/indiicator_data.csv', index=True)

In [ ]:
df_indicator.head(5)

,province,month,disease,case_type,age_group,indicator,value
0,Zambia Ministry of Health,2021-01-01,cholera,reported,All ages,Cholera 1-4 years CFR (%),NaN
1,Zambia Ministry of Health,2021-01-01,cholera,reported,All ages,Cholera 1-4 years incidence (1K),NaN
2,Zambia Ministry of Health,2021-01-01,cholera,reported,5 years and older,Cholera 5 years and older CFR (%),NaN
3,Zambia Ministry of Health,2021-01-01,cholera,reported,5 years and older,Cholera 5 years and older incidence (1K),0.0
4,Zambia Ministry of Health,2021-01-01,cholera,reported,5 years and older,Cholera 5 years and older - total,2.0
